# Vector embedding

Notebook này được chỉnh theo đúng dữ liệu hiện tại:
- Model: SigLIP2
- **Nguồn ảnh:** dùng trực tiếp keyframe đã gắn vào Kaggle, dạng `001_Keyframes_L21/keyframes/L21_V001/001.jpg`.
- **Batch xử lý:** `L21` → `L25`
- **Model:** `google/siglip2-so400m-patch14-384`
- **CPU:** một `DataLoader` dài hạn trên mỗi GPU, worker tự điều chỉnh theo số CPU, `pin_memory`, prefetch và batched image preprocessing.
- **Inference:** FP16 trên GPU, sau đó L2-normalize ở FP32.
- **Output GCS:** giữ hierarchy `features/extractors/.../extractor=vector-embedding/...` để đồng nhất với `captioning`, `object_detection`, `ocr`.
- Mỗi video có một `.npy` và một CSV mapping cùng số dòng.

> Ghi chú: vì dataset Kaggle bạn gửi chỉ chứa file keyframe, CSV mapping ở đây dùng thứ tự file local (`001.jpg`, `002.jpg`, ...). Notebook **không giả định timestamp/FPS** nếu metadata đó không có trong dataset local.


## 1. Install dependencies

In [1]:
%pip install -q -U google-cloud-storage pandas pillow tqdm transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.5/341.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 92.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 105.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, whi

## 2. Configuration

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GCS_BUCKET")
secret_value_1 = user_secrets.get_secret("GCS_CREDENTIALS_JSON")
secret_value_2 = user_secrets.get_secret("HF_TOKEN")

In [7]:
from pathlib import Path
import json
import os
import torch

# -------------------------
# Local Kaggle input
# -------------------------
# Có thể để /kaggle/input; code sẽ tự tìm:
# <dataset>/001_Keyframes_L21/keyframes/
# <dataset>/002_Keyframes_L22/keyframes/
# ...
INPUT_ROOT = "/kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25"

# BATCHES = ["L21", "L22", "L23", "L24", "L25"]
BATCHES = ["L22"]

# None = chạy toàn bộ video.
# Ví dụ đặt 1 để smoke-test trước khi chạy full.
MAX_VIDEOS_PER_BATCH = None

# -------------------------
# SigLIP2
# -------------------------
CHECKPOINT = "google/siglip2-so400m-patch14-384"
EMBEDDING_DIM = 1152

# T4 16 GB: 512/image batch per GPU thường còn khá dư VRAM với model Base + FP16.
# Nếu môi trường cụ thể OOM, giảm xuống 384 hoặc 256.
BATCH_SIZE_PER_GPU = 512

# None = tự động chọn theo CPU / số GPU.
NUM_WORKERS_PER_GPU = None
PREFETCH_FACTOR = 2

# Compute vẫn FP16; lưu float32 để an toàn cho retrieval/indexing.
# Có thể đổi thành "float16" nếu muốn giảm 50% dung lượng GCS.
SAVE_DTYPE = "float32"

# -------------------------
# GCS output
# -------------------------
GCS_BUCKET = "aic_ai_2026"
DATASET_ID = "ai_challenge_2025"
FRAME_PROFILE = "autoshot_v1"

OUTPUT_PREFIX = "features/extractors"

# Stable output prefix giúp resume qua nhiều Kaggle session.
EXTRACTOR_VERSION = "siglip2-so400m-patch14-384-v1"

UPLOAD_TO_GCS = True
SKIP_EXISTING = True
UPLOAD_WORKERS_PER_RANK = 4

# Cách 1: để JSON service account trong Kaggle Secret.
GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"

# Cách 2: hoặc trỏ trực tiếp tới file credentials JSON đã attach vào Kaggle.
# Nếu dùng Secret thì để chuỗi rỗng.
GCS_CREDENTIALS_FILE = ""

LOCAL_OUTPUT_ROOT = "/kaggle/working/vector_embedding_siglip2"

NPROC = min(2, torch.cuda.device_count())

print("CUDA count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} - {torch.cuda.get_device_name(i)}")
print("torchrun processes:", NPROC)

print("Batches:", BATCHES)


CUDA count: 2
  cuda:0 - Tesla T4
  cuda:1 - Tesla T4
torchrun processes: 2
Batches: ['L22']


## 3. Preview local folders before embedding

Cell này **chỉ nhìn local Kaggle input**, không gọi GCS và không download ảnh.


In [8]:
from pathlib import Path

root = Path(INPUT_ROOT)
print(root)

def find_batch_dirs(batch):
    candidates = []
    candidates += list(root.glob(f"*Keyframes_{batch}/keyframes"))
    candidates += list(root.glob(f"*/*Keyframes_{batch}/keyframes"))
    candidates += list(root.glob(f"*/*/*Keyframes_{batch}/keyframes"))
    return [p for p in candidates if p.is_dir()]

# Count the number of videos and frames
for batch in BATCHES:
    dirs = find_batch_dirs(batch)
    print(f"\n{batch}:")
    for d in dirs:
        videos = [p for p in d.iterdir() if p.is_dir() and p.name.startswith(batch + "_V")]
        image_count = 0
        for v in videos:
            image_count += sum(
                1 for p in v.iterdir()
                if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
            )
        print(" ", d)
        print("    videos =", len(videos), "| images =", image_count)


/kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25

L22:
  /kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25/002_Keyframes_L22/keyframes
    videos = 31 | images = 9096


## 4. Write the optimized 2-GPU worker script

Vì bài toán này chỉ là inference độc lập giữa các frame/video, không cần gradient synchronization. `torchrun` được dùng để tạo **một process/model trên mỗi T4**, rồi chia video theo tổng số frame để cân bằng tải.


In [9]:
SCRIPT_PATH = '/kaggle/working/embed_siglip_kaggle_t4x2.py'
script_text = 'from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport os\nimport re\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\nfrom tqdm.auto import tqdm\n\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import AutoImageProcessor, AutoModel\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\nVALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}\n\n\n@dataclass(frozen=True)\nclass VideoJob:\n    batch_id: str\n    video_id: str\n    video_dir: str\n    image_paths: tuple[str, ...]\n\n    @property\n    def num_frames(self) -> int:\n        return len(self.image_paths)\n\n\ndef natural_key(path: str | Path):\n    name = Path(path).stem\n    if name.isdigit():\n        return (0, int(name))\n    parts = re.split(r"(\\d+)", name)\n    return (1, tuple(int(p) if p.isdigit() else p.lower() for p in parts))\n\n\ndef init_distributed():\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU is required for this notebook.")\n\n    torch.cuda.set_device(local_rank)\n    if world_size > 1 and not dist.is_initialized():\n        dist.init_process_group(backend="nccl")\n\n    device = torch.device(f"cuda:{local_rank}")\n    return rank, local_rank, world_size, device\n\n\ndef destroy_distributed():\n    if dist.is_available() and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef barrier():\n    if dist.is_available() and dist.is_initialized():\n        dist.barrier()\n\n\ndef broadcast_object(obj, rank: int):\n    if not (dist.is_available() and dist.is_initialized()):\n        return obj\n    container = [obj if rank == 0 else None]\n    dist.broadcast_object_list(container, src=0)\n    return container[0]\n\n\ndef read_kaggle_secret(secret_name: str) -> str:\n    if not secret_name:\n        return ""\n    try:\n        from kaggle_secrets import UserSecretsClient\n        return UserSecretsClient().get_secret(secret_name) or ""\n    except Exception:\n        return ""\n\n\ndef make_storage_client(cfg: dict[str, Any]):\n    from google.cloud import storage\n\n    cred_file = str(cfg.get("gcs_credentials_file", "") or "").strip()\n    secret_name = str(cfg.get("gcs_credentials_json_secret_name", "") or "").strip()\n    cred_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()\n\n    if not cred_json and secret_name:\n        cred_json = read_kaggle_secret(secret_name).strip()\n\n    if cred_json:\n        from google.oauth2 import service_account\n        info = json.loads(cred_json)\n        credentials = service_account.Credentials.from_service_account_info(info)\n        return storage.Client(project=credentials.project_id, credentials=credentials)\n\n    if cred_file:\n        return storage.Client.from_service_account_json(cred_file)\n\n    return storage.Client()\n\n\ndef batch_output_prefix(cfg: dict[str, Any], batch_id: str) -> str:\n    return (\n        f"{cfg[\'output_prefix\'].strip(\'/\')}/"\n        f"dataset={cfg[\'dataset_id\']}/"\n        f"batch={batch_id}/"\n        f"frame_profile={cfg[\'frame_profile\']}/"\n        f"extractor=vector-embedding/"\n        f"extractor_version={cfg[\'extractor_version\']}"\n    )\n\n\ndef discover_keyframes_dir(input_root: Path, batch_id: str) -> Path:\n    candidates: list[Path] = []\n\n    # Case 1: INPUT_ROOT is already the Kaggle dataset root.\n    candidates.extend(input_root.glob(f"*Keyframes_{batch_id}/keyframes"))\n\n    # Case 2: INPUT_ROOT=/kaggle/input and the first level is the Kaggle dataset slug.\n    candidates.extend(input_root.glob(f"*/*Keyframes_{batch_id}/keyframes"))\n\n    # Fallback for one extra nesting level without scanning every image recursively.\n    candidates.extend(input_root.glob(f"*/*/*Keyframes_{batch_id}/keyframes"))\n\n    unique = []\n    seen = set()\n    for p in candidates:\n        try:\n            resolved = p.resolve()\n        except Exception:\n            resolved = p\n        if str(resolved) not in seen and p.is_dir():\n            seen.add(str(resolved))\n            unique.append(p)\n\n    valid = []\n    for p in unique:\n        has_video = any(\n            child.is_dir() and child.name.startswith(batch_id + "_V")\n            for child in p.iterdir()\n        )\n        if has_video:\n            valid.append(p)\n\n    if not valid:\n        raise FileNotFoundError(\n            f"Cannot find local keyframes for {batch_id} under {input_root}. "\n            f"Expected a path like <dataset>/001_Keyframes_{batch_id}/keyframes/."\n        )\n\n    # Prefer the shallowest candidate.\n    valid.sort(key=lambda p: (len(p.parts), str(p)))\n    if len(valid) > 1:\n        print(f"[discover] Multiple candidates for {batch_id}; using {valid[0]}")\n        for extra in valid[1:]:\n            print(f"           ignored: {extra}")\n    return valid[0]\n\n\ndef discover_jobs(cfg: dict[str, Any]) -> list[VideoJob]:\n    input_root = Path(cfg["input_root"])\n    jobs: list[VideoJob] = []\n    max_videos = cfg.get("max_videos_per_batch")\n\n    for batch_id in cfg["batches"]:\n        keyframes_dir = discover_keyframes_dir(input_root, batch_id)\n        video_dirs = sorted(\n            [p for p in keyframes_dir.iterdir() if p.is_dir() and p.name.startswith(batch_id + "_V")],\n            key=lambda p: natural_key(p.name),\n        )\n        if max_videos is not None:\n            video_dirs = video_dirs[: int(max_videos)]\n\n        for video_dir in video_dirs:\n            image_paths = sorted(\n                [p for p in video_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS],\n                key=natural_key,\n            )\n            if not image_paths:\n                continue\n            jobs.append(\n                VideoJob(\n                    batch_id=batch_id,\n                    video_id=video_dir.name,\n                    video_dir=str(video_dir),\n                    image_paths=tuple(str(p) for p in image_paths),\n                )\n            )\n\n    jobs.sort(key=lambda j: (j.batch_id, natural_key(j.video_id)))\n    return jobs\n\n\ndef get_existing_complete_videos(cfg: dict[str, Any], jobs: list[VideoJob]) -> dict[str, set[str]]:\n    if not cfg.get("upload_to_gcs", True) or not cfg.get("skip_existing", True):\n        return {batch: set() for batch in cfg["batches"]}\n\n    client = make_storage_client(cfg)\n    bucket = client.bucket(cfg["gcs_bucket"])\n    result: dict[str, set[str]] = {}\n\n    for batch_id in cfg["batches"]:\n        prefix = batch_output_prefix(cfg, batch_id) + "/"\n        emb_prefix = prefix + "embeddings/"\n        map_prefix = prefix + "map-keyframes/"\n\n        emb = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=emb_prefix)\n            if blob.name.endswith(".npy")\n        }\n        maps = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=map_prefix)\n            if blob.name.endswith(".csv")\n        }\n        discovered_ids = {j.video_id for j in jobs if j.batch_id == batch_id}\n        result[batch_id] = (emb & maps) & discovered_ids\n\n    return result\n\n\ndef greedy_partition(jobs: list[VideoJob], world_size: int) -> list[list[VideoJob]]:\n    partitions: list[list[VideoJob]] = [[] for _ in range(world_size)]\n    loads = [0] * world_size\n\n    # Largest videos first, then greedily place onto the least-loaded GPU.\n    for job in sorted(jobs, key=lambda j: (-j.num_frames, j.batch_id, j.video_id)):\n        idx = min(range(world_size), key=lambda r: (loads[r], r))\n        partitions[idx].append(job)\n        loads[idx] += job.num_frames\n\n    for part in partitions:\n        part.sort(key=lambda j: (j.batch_id, natural_key(j.video_id)))\n    return partitions\n\n\ndef frame_metadata(job: VideoJob, image_path: str, row_number: int) -> dict[str, Any]:\n    p = Path(image_path)\n    stem = p.stem\n    keyframe_number = int(stem) if stem.isdigit() else row_number\n    return {\n        "n": row_number,\n        "keyframe_number": keyframe_number,\n        "keyframe_id": f"{job.video_id}_{stem}",\n        "frame_filename": p.name,\n        "image_rel_path": f"{job.video_id}/{p.name}",\n    }\n\n\nclass LocalFrameDataset(Dataset):\n    def __init__(self, records: list[dict[str, Any]]):\n        self.records = records\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, idx):\n        path = self.records[idx]["image_path"]\n        try:\n            with Image.open(path) as image:\n                image = image.convert("RGB").copy()\n        except Exception as exc:\n            raise RuntimeError(f"Failed to read image: {path}") from exc\n        return image, idx\n\n\nclass ProcessorCollator:\n    def __init__(self, image_processor):\n        self.image_processor = image_processor\n\n    def __call__(self, batch):\n        images, indices = zip(*batch)\n        inputs = self.image_processor(images=list(images), return_tensors="pt")\n        return dict(inputs), torch.tensor(indices, dtype=torch.int64)\n\n\ndef extract_embedding_tensor(outputs):\n    if torch.is_tensor(outputs):\n        return outputs\n    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:\n        return outputs.pooler_output\n    if hasattr(outputs, "image_embeds") and outputs.image_embeds is not None:\n        return outputs.image_embeds\n    raise TypeError(f"Unsupported get_image_features output type: {type(outputs)}")\n\n\ndef atomic_save_npy(path: Path, array: np.ndarray):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with tmp.open("wb") as f:\n        np.save(f, array)\n    os.replace(tmp, path)\n\n\ndef save_video_outputs(\n    cfg: dict[str, Any],\n    local_root: Path,\n    job: VideoJob,\n    embedding_chunks: list[np.ndarray],\n):\n    embeddings = np.concatenate(embedding_chunks, axis=0)\n    if embeddings.shape[0] != job.num_frames:\n        raise RuntimeError(\n            f"{job.video_id}: embedding rows={embeddings.shape[0]} "\n            f"but local frames={job.num_frames}"\n        )\n\n    save_dtype = np.float16 if cfg["save_dtype"] == "float16" else np.float32\n    embeddings = embeddings.astype(save_dtype, copy=False)\n\n    batch_dir = local_root / job.batch_id\n    emb_path = batch_dir / "embeddings" / f"{job.video_id}.npy"\n    map_path = batch_dir / "map-keyframes" / f"{job.video_id}.csv"\n\n    atomic_save_npy(emb_path, embeddings)\n\n    rows = [\n        frame_metadata(job, image_path, i + 1)\n        for i, image_path in enumerate(job.image_paths)\n    ]\n    map_path.parent.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(rows).to_csv(map_path, index=False)\n\n    # Validate row alignment immediately.\n    if len(rows) != embeddings.shape[0]:\n        raise RuntimeError(f"{job.video_id}: map/embedding row mismatch")\n\n    return emb_path, map_path, embeddings.shape\n\n\ndef build_records(jobs: list[VideoJob]) -> tuple[list[dict[str, Any]], dict[str, VideoJob]]:\n    records: list[dict[str, Any]] = []\n    job_by_video: dict[str, VideoJob] = {}\n\n    for job in jobs:\n        if job.video_id in job_by_video:\n            raise ValueError(f"Duplicate video_id across selected data: {job.video_id}")\n        job_by_video[job.video_id] = job\n        for image_path in job.image_paths:\n            records.append(\n                {\n                    "batch_id": job.batch_id,\n                    "video_id": job.video_id,\n                    "image_path": image_path,\n                }\n            )\n    return records, job_by_video\n\n\ndef upload_one(bucket, local_path: Path, object_key: str):\n    suffix = local_path.suffix.lower()\n    content_type = {\n        ".csv": "text/csv",\n        ".json": "application/json",\n        ".npy": "application/octet-stream",\n    }.get(suffix, "application/octet-stream")\n    bucket.blob(object_key).upload_from_filename(\n        str(local_path),\n        content_type=content_type,\n        timeout=900,\n    )\n\n\ndef upload_rank_outputs(cfg: dict[str, Any], paths: list[tuple[str, Path]]):\n    if not cfg.get("upload_to_gcs", True) or not paths:\n        return\n\n    client = make_storage_client(cfg)\n    bucket = client.bucket(cfg["gcs_bucket"])\n    workers = max(1, int(cfg.get("upload_workers_per_rank", 4)))\n\n    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-upload") as pool:\n        futures = {\n            pool.submit(upload_one, bucket, local_path, object_key): (local_path, object_key)\n            for object_key, local_path in paths\n        }\n        for future in tqdm(\n            as_completed(futures),\n            total=len(futures),\n            desc="Uploading rank outputs",\n            leave=False,\n        ):\n            future.result()\n\n\ndef write_json(path: Path, payload: dict[str, Any]):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef finalize_batch_metadata(\n    cfg: dict[str, Any],\n    local_root: Path,\n    jobs_all: list[VideoJob],\n    existing: dict[str, set[str]],\n    elapsed: float,\n):\n    client = make_storage_client(cfg) if cfg.get("upload_to_gcs", True) else None\n    bucket = client.bucket(cfg["gcs_bucket"]) if client else None\n\n    for batch_id in cfg["batches"]:\n        batch_jobs = [j for j in jobs_all if j.batch_id == batch_id]\n        batch_dir = local_root / batch_id\n        batch_dir.mkdir(parents=True, exist_ok=True)\n\n        produced_embs = sorted((batch_dir / "embeddings").glob("*.npy")) if (batch_dir / "embeddings").exists() else []\n        produced_maps = sorted((batch_dir / "map-keyframes").glob("*.csv")) if (batch_dir / "map-keyframes").exists() else []\n\n        model_info = {\n            "checkpoint": cfg["checkpoint"],\n            "embedding_dimension": int(cfg["embedding_dim"]),\n            "normalized": True,\n            "normalization": "L2",\n            "inference_dtype": "float16",\n            "saved_dtype": cfg["save_dtype"],\n            "image_source": "Kaggle local dataset; no frame download from GCS",\n            "multi_gpu_strategy": "torchrun one independent inference process per GPU",\n            "extractor": "vector-embedding",\n            "extractor_version": cfg["extractor_version"],\n        }\n\n        summary = {\n            "status": "PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "SUCCESS",\n            "batch_id": batch_id,\n            "videos_discovered": len(batch_jobs),\n            "frames_discovered": int(sum(j.num_frames for j in batch_jobs)),\n            "videos_skipped_existing_gcs": len(existing.get(batch_id, set())),\n            "videos_produced_this_session": len(produced_embs),\n            "maps_produced_this_session": len(produced_maps),\n            "elapsed_seconds_global": round(elapsed, 3),\n            "gcs_prefix": f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/",\n        }\n\n        model_info_path = batch_dir / "model_info.json"\n        summary_path = batch_dir / "summary.json"\n        write_json(model_info_path, model_info)\n        write_json(summary_path, summary)\n\n        if bucket is not None:\n            prefix = batch_output_prefix(cfg, batch_id) + "/"\n            upload_one(bucket, model_info_path, prefix + "model_info.json")\n            upload_one(bucket, summary_path, prefix + "summary.json")\n            marker = "_PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "_SUCCESS"\n            bucket.blob(prefix + marker).upload_from_string("", content_type="text/plain")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n\n    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    rank, local_rank, world_size, device = init_distributed()\n\n    cpu_count = os.cpu_count() or 4\n    configured_workers = cfg.get("num_workers_per_gpu")\n    if configured_workers is None:\n        workers = max(1, min(4, cpu_count // max(1, world_size)))\n    else:\n        workers = max(0, int(configured_workers))\n\n    # Avoid excessive intra-op CPU thread competition with DataLoader workers.\n    torch.set_num_threads(1)\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n\n    if rank == 0:\n        print(f"GPUs: {world_size}")\n        for i in range(torch.cuda.device_count()):\n            print(f"  cuda:{i}: {torch.cuda.get_device_name(i)}")\n        print(f"CPU cores visible: {cpu_count}")\n        print(f"DataLoader workers per GPU: {workers}")\n\n    all_jobs = discover_jobs(cfg)\n\n    if rank == 0:\n        per_batch = {}\n        for batch in cfg["batches"]:\n            bj = [j for j in all_jobs if j.batch_id == batch]\n            per_batch[batch] = {\n                "videos": len(bj),\n                "frames": int(sum(j.num_frames for j in bj)),\n            }\n        print("Local discovery:", json.dumps(per_batch, indent=2))\n\n    existing = None\n    if rank == 0:\n        existing = get_existing_complete_videos(cfg, all_jobs)\n        if cfg.get("skip_existing", True):\n            print("Existing complete videos on GCS:", {k: len(v) for k, v in existing.items()})\n    existing = broadcast_object(existing, rank)\n\n    remaining_jobs = [\n        j for j in all_jobs\n        if j.video_id not in existing.get(j.batch_id, set())\n    ]\n\n    partitions = greedy_partition(remaining_jobs, world_size)\n    my_jobs = partitions[rank]\n\n    if rank == 0:\n        loads = [sum(j.num_frames for j in p) for p in partitions]\n        print("Planned frames per GPU:", loads)\n\n    my_frames = sum(j.num_frames for j in my_jobs)\n    print(\n        f"[rank {rank}] cuda:{local_rank} | videos={len(my_jobs)} "\n        f"| frames={my_frames}"\n    )\n\n    if not my_jobs:\n        barrier()\n        if rank == 0:\n            local_root = Path(cfg["local_output_root"])\n            finalize_batch_metadata(cfg, local_root, all_jobs, existing, elapsed=0.0)\n        barrier()\n        destroy_distributed()\n        return\n\n    torch.cuda.reset_peak_memory_stats(device)\n    started = time.perf_counter()\n\n    # Image-only processor avoids loading the tokenizer on every rank/worker.\n    image_processor = AutoImageProcessor.from_pretrained(\n        cfg["checkpoint"],\n        use_fast=True,\n    )\n    model = AutoModel.from_pretrained(\n        cfg["checkpoint"],\n        dtype=torch.float16,\n    ).to(device).eval()\n\n    records, job_by_video = build_records(my_jobs)\n    dataset = LocalFrameDataset(records)\n\n    loader_kwargs = dict(\n        dataset=dataset,\n        batch_size=int(cfg["batch_size_per_gpu"]),\n        shuffle=False,\n        num_workers=workers,\n        pin_memory=True,\n        persistent_workers=(workers > 0),\n        collate_fn=ProcessorCollator(image_processor),\n        drop_last=False,\n    )\n    if workers > 0:\n        loader_kwargs["prefetch_factor"] = int(cfg.get("prefetch_factor", 2))\n\n    loader = DataLoader(**loader_kwargs)\n    local_root = Path(cfg["local_output_root"])\n    local_root.mkdir(parents=True, exist_ok=True)\n\n    current_video = None\n    current_chunks: list[np.ndarray] = []\n    created_uploads: list[tuple[str, Path]] = []\n    processed_videos = 0\n    processed_frames = 0\n\n    def flush_current():\n        nonlocal current_video, current_chunks, processed_videos, processed_frames\n        if current_video is None:\n            return\n        job = job_by_video[current_video]\n        emb_path, map_path, shape = save_video_outputs(\n            cfg=cfg,\n            local_root=local_root,\n            job=job,\n            embedding_chunks=current_chunks,\n        )\n        prefix = batch_output_prefix(cfg, job.batch_id) + "/"\n        created_uploads.append((prefix + f"embeddings/{job.video_id}.npy", emb_path))\n        created_uploads.append((prefix + f"map-keyframes/{job.video_id}.csv", map_path))\n        processed_videos += 1\n        processed_frames += shape[0]\n        current_video = None\n        current_chunks = []\n\n    pbar = tqdm(\n        loader,\n        total=len(loader),\n        desc=f"rank{rank} cuda:{local_rank}",\n        position=rank,\n        leave=True,\n    )\n\n    with torch.inference_mode():\n        for inputs, sample_indices in pbar:\n            gpu_inputs = {\n                k: v.to(device, non_blocking=True)\n                for k, v in inputs.items()\n            }\n\n            with torch.autocast(\n                device_type="cuda",\n                dtype=torch.float16,\n                enabled=True,\n            ):\n                outputs = model.get_image_features(**gpu_inputs)\n                emb = extract_embedding_tensor(outputs)\n\n            emb = F.normalize(emb.float(), p=2, dim=-1)\n            emb_np = emb.cpu().numpy()\n\n            indices = sample_indices.tolist()\n            batch_video_ids = [records[i]["video_id"] for i in indices]\n\n            start = 0\n            while start < len(indices):\n                vid = batch_video_ids[start]\n                end = start + 1\n                while end < len(indices) and batch_video_ids[end] == vid:\n                    end += 1\n\n                if current_video is None:\n                    current_video = vid\n                elif vid != current_video:\n                    flush_current()\n                    current_video = vid\n\n                current_chunks.append(emb_np[start:end])\n                start = end\n\n    flush_current()\n    torch.cuda.synchronize(device)\n\n    upload_started = time.perf_counter()\n    upload_rank_outputs(cfg, created_uploads)\n    upload_seconds = time.perf_counter() - upload_started\n\n    elapsed = time.perf_counter() - started\n    peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2\n\n    rank_metrics = {\n        "rank": rank,\n        "local_rank": local_rank,\n        "gpu": torch.cuda.get_device_name(device),\n        "videos_processed": processed_videos,\n        "frames_processed": processed_frames,\n        "elapsed_seconds": round(elapsed, 3),\n        "upload_seconds": round(upload_seconds, 3),\n        "frames_per_second_including_upload": round(processed_frames / elapsed, 3) if elapsed else 0,\n        "peak_allocated_mb": round(peak_mb, 2),\n        "batch_size_per_gpu": int(cfg["batch_size_per_gpu"]),\n        "num_workers": workers,\n    }\n    write_json(local_root / f"rank_{rank}_metrics.json", rank_metrics)\n    print(f"[rank {rank}] metrics:", json.dumps(rank_metrics, indent=2))\n\n    barrier()\n\n    if rank == 0:\n        global_elapsed = elapsed\n        metric_files = sorted(local_root.glob("rank_*_metrics.json"))\n        if metric_files:\n            metrics = [json.loads(p.read_text(encoding="utf-8")) for p in metric_files]\n            global_elapsed = max(float(m["elapsed_seconds"]) for m in metrics)\n            write_json(local_root / "all_rank_metrics.json", {"ranks": metrics})\n\n        finalize_batch_metadata(\n            cfg=cfg,\n            local_root=local_root,\n            jobs_all=all_jobs,\n            existing=existing,\n            elapsed=global_elapsed,\n        )\n\n        print("\\nFinal GCS prefixes:")\n        for batch_id in cfg["batches"]:\n            print(f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/")\n\n    barrier()\n    destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n'
Path(SCRIPT_PATH).write_text(script_text, encoding='utf-8')
print('Wrote:', SCRIPT_PATH, '| bytes:', Path(SCRIPT_PATH).stat().st_size)


Wrote: /kaggle/working/embed_siglip_kaggle_t4x2.py | bytes: 23311


## 5. Build runtime config

In [10]:
CONFIG_PATH = "/kaggle/working/siglip2_embedding_config.json"

cfg = {
    "input_root": INPUT_ROOT,
    "batches": BATCHES,
    "max_videos_per_batch": MAX_VIDEOS_PER_BATCH,

    "checkpoint": CHECKPOINT,
    "embedding_dim": EMBEDDING_DIM,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
    "num_workers_per_gpu": NUM_WORKERS_PER_GPU,
    "prefetch_factor": PREFETCH_FACTOR,
    "save_dtype": SAVE_DTYPE,

    "gcs_bucket": GCS_BUCKET,
    "dataset_id": DATASET_ID,
    "frame_profile": FRAME_PROFILE,
    "output_prefix": OUTPUT_PREFIX,
    "extractor_version": EXTRACTOR_VERSION,

    "upload_to_gcs": UPLOAD_TO_GCS,
    "skip_existing": SKIP_EXISTING,
    "upload_workers_per_rank": UPLOAD_WORKERS_PER_RANK,
    "gcs_credentials_json_secret_name": GCS_CREDENTIALS_JSON_SECRET_NAME,
    "gcs_credentials_file": GCS_CREDENTIALS_FILE,

    "local_output_root": LOCAL_OUTPUT_ROOT,
}

Path(CONFIG_PATH).write_text(
    json.dumps(cfg, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(Path(CONFIG_PATH).read_text())


{
  "input_root": "/kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25",
  "batches": [
    "L22"
  ],
  "max_videos_per_batch": null,
  "checkpoint": "google/siglip2-so400m-patch14-384",
  "embedding_dim": 1152,
  "batch_size_per_gpu": 512,
  "num_workers_per_gpu": null,
  "prefetch_factor": 2,
  "save_dtype": "float32",
  "gcs_bucket": "aic_ai_2026",
  "dataset_id": "ai_challenge_2025",
  "frame_profile": "autoshot_v1",
  "output_prefix": "features/extractors",
  "extractor_version": "siglip2-so400m-patch14-384-v1",
  "upload_to_gcs": true,
  "skip_existing": true,
  "upload_workers_per_rank": 4,
  "gcs_credentials_json_secret_name": "GCS_CREDENTIALS_JSON",
  "gcs_credentials_file": "",
  "local_output_root": "/kaggle/working/vector_embedding_siglip2"
}


## 6. Smoke test (recommended)

Để test nhanh trước:

1. đổi `MAX_VIDEOS_PER_BATCH = 1` ở config,
2. chạy lại cell config,
3. chạy cell `torchrun` bên dưới.

Sau khi ổn, đổi về `None` và chạy full. Với `SKIP_EXISTING=True`, video đã có đủ `.npy` + `.csv` trên GCS sẽ được bỏ qua. Smoke test chỉ ghi `_PARTIAL_SUCCESS`; `_SUCCESS` chỉ được ghi khi `MAX_VIDEOS_PER_BATCH=None`.


## 7. Run embedding on T4×2

In [11]:
import os
import subprocess

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    f"--nproc_per_node={NPROC}",
    SCRIPT_PATH,
    "--config",
    CONFIG_PATH,
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, env=env, check=True)


Running: torchrun --standalone --nproc_per_node=2 /kaggle/working/embed_siglip_kaggle_t4x2.py --config /kaggle/working/siglip2_embedding_config.json


W0818 08:01:42.347000 187 torch/distributed/run.py:852] 
W0818 08:01:42.347000 187 torch/distributed/run.py:852] *****************************************
W0818 08:01:42.347000 187 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0818 08:01:42.347000 187 torch/distributed/run.py:852] *****************************************
[W818 08:01:42.819494222 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W818 08:02:04.409480462 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W818 08:02:04.410047123 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


GPUs: 2
  cuda:0: Tesla T4
  cuda:1: Tesla T4
CPU cores visible: 4
DataLoader workers per GPU: 2
Local discovery: {
  "L22": {
    "videos": 31,
    "frames": 9096
  }
}
Existing complete videos on GCS: {'L22': 0}
Planned frames per GPU: [4424, 4672]
[rank 0] cuda:0 | videos=15 | frames=4424
[rank 1] cuda:1 | videos=16 | frames=4672


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between

[rank 0] metrics: {
  "rank": 0,
  "local_rank": 0,
  "gpu": "Tesla T4",
  "videos_processed": 15,
  "frames_processed": 4424,
  "elapsed_seconds": 136.105,
  "upload_seconds": 11.502,
  "frames_per_second_including_upload": 32.504,
  "peak_allocated_mb": 4006.21,
  "batch_size_per_gpu": 512,
  "num_workers": 2
}


[rank 1] metrics: {
  "rank": 1,
  "local_rank": 1,
  "gpu": "Tesla T4",
  "videos_processed": 16,
  "frames_processed": 4672,
  "elapsed_seconds": 146.777,
  "upload_seconds": 18.359,
  "frames_per_second_including_upload": 31.831,
  "peak_allocated_mb": 4006.21,
  "batch_size_per_gpu": 512,
  "num_workers": 2
}

Final GCS prefixes:
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L22/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m-patch14-384-v1/


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '/kaggle/working/embed_siglip_kaggle_t4x2.py', '--config', '/kaggle/working/siglip2_embedding_config.json'], returncode=0)

## 8. Validate local output

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(LOCAL_OUTPUT_ROOT)

total_videos = 0
total_vectors = 0

for batch in BATCHES:
    emb_dir = root / batch / "embeddings"
    map_dir = root / batch / "map-keyframes"

    npy_files = sorted(emb_dir.glob("*.npy")) if emb_dir.exists() else []
    print(f"\n{batch}: produced this session = {len(npy_files)} videos")

    for npy_path in npy_files:
        csv_path = map_dir / f"{npy_path.stem}.csv"
        assert csv_path.exists(), f"Missing map CSV: {csv_path}"

        emb = np.load(npy_path, mmap_mode="r")
        df = pd.read_csv(csv_path)

        assert emb.ndim == 2
        assert emb.shape[0] == len(df), (npy_path.name, emb.shape, len(df))
        assert emb.shape[1] == EMBEDDING_DIM, (npy_path.name, emb.shape)

        total_videos += 1
        total_vectors += emb.shape[0]

print("\nValidated videos:", total_videos)
print("Validated vectors:", total_vectors)



L22: produced this session = 31 videos

Validated videos: 31
Validated vectors: 9096


## 9. Expected GCS layout

In [13]:
for batch in BATCHES:
    prefix = (
        f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/"
        f"dataset={DATASET_ID}/batch={batch}/"
        f"frame_profile={FRAME_PROFILE}/"
        f"extractor=vector-embedding/"
        f"extractor_version={EXTRACTOR_VERSION}/"
    )
    print(prefix)

print(
    "\nInside each batch prefix:\n"
    "  embeddings/Lxx_Vxxx.npy\n"
    "  map-keyframes/Lxx_Vxxx.csv\n"
    "  model_info.json\n"
    "  summary.json\n"
    "  _SUCCESS  # full run; smoke test dùng _PARTIAL_SUCCESS"
)


gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L22/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m-patch14-384-v1/

Inside each batch prefix:
  embeddings/Lxx_Vxxx.npy
  map-keyframes/Lxx_Vxxx.csv
  model_info.json
  summary.json
  _SUCCESS  # full run; smoke test dùng _PARTIAL_SUCCESS


## Tuning nhanh cho Kaggle T4×2

- Bắt đầu với `BATCH_SIZE_PER_GPU=512`.
- Nếu OOM: giảm `512 → 384 → 256`.
- `NUM_WORKERS_PER_GPU=None` sẽ tự chọn khoảng `CPU cores / 2 GPUs`, tối đa 4 worker/GPU.
- Không tăng worker quá cao nếu CPU chỉ có 4 core; GPU có thể nhanh hơn nhưng context switching và RAM prefetch sẽ tăng.
- `SAVE_DTYPE="float32"` giữ vector an toàn hơn cho indexing; `float16` giảm một nửa dung lượng upload/storage.
- Pipeline chỉ upload **sau khi embedding**, nên GCS network không tranh CPU/I/O với phần inference chính.
